# Prompt Chaining with LangChain - A Comprehensive Overview

Prompt chaining is a foundational concept in building advanced workflows using language models (LLMs). It involves linking multiple prompts in a logical sequence, where the output of one prompt serves as the input for the next. This modular approach is powerful for solving complex tasks like multi-step text processing, summarization, question-answering, and more.

**LangChain** is a versatile framework designed to simplify the creation of such workflows. It provides tools to manage LLMs, define custom prompts, and connect them into reusable chains. By abstracting the complexity of managing prompts, LangChain allows developers to focus on solving problems rather than orchestrating interactions with LLMs.

In this tutorial, we will:
1. Explore different types of prompt chaining (Sequential, Branching, Iterative, etc.).
Implement a generic chaining example combining Sequential, Branching, and Iterative chaining types.
Leverage LangChain's built-in classes like `PromptTemplate`, `LLMChain`, and `SequentialChain` to define and manage the workflow.

## How LangChain Manages Prompt Chaining

- **Prompt Abstraction:** LangChain uses PromptTemplate to define input-output structures for each step.
- **LLM Integration:** The framework supports multiple LLMs, including OpenAI, Granite, Hugging Face, etc.
- **Chain Management:** LangChain's SequentialChain and SimpleSequentialChain allow for linking steps in a defined sequence.
- **Dynamic Workflows:** With tools like ConditionalChain, LangChain can adapt workflows based on intermediate outputs.

By the end of this tutorial, you’ll have a solid understanding of how to use LangChain to build modular and extensible workflows for a wide range of applications.

## Types of Prompt Chaining 

Prompt chaining allows you to design workflows where outputs from one step are passed to the next. Different types of chaining support diverse workflows, ranging from simple sequential tasks to more complex, dynamic processes. Here’s a brief look at the types of prompt chaining:

- **Sequential Chaining:** The most straightforward type of chaining, where the output of one prompt is directly passed as input to the next. This is ideal for tasks with a linear progression. 
- **Branching Chaining:** In branching chaining, a single output is split into multiple parallel workflows. Each branch processes the output independently.
- **Iterative Chaining:** Iterative chaining involves repeatedly running a prompt or chain until a specified condition is met. This is useful for refining outputs.
- **Hierarchical Chaining:** This type breaks down a large task into smaller subtasks, which are executed hierarchically. Lower-level outputs feed higher-level tasks.
- **Conditional Chaining:** Conditional chaining dynamically chooses the next step based on the output of a prior prompt. It enables decision-making within workflows.
- **Multi-Modal Chaining:** Multi-modal chaining integrates prompts that handle different data types (e.g., text, images, or audio). It is suitable for applications combining multiple modalities.
- **Dynamic Chaining:** Dynamic chaining adapts the workflow based on real-time outputs or changing conditions. It adds flexibility to prompt chaining.
- **Recursive Chaining:** In recursive chaining, large inputs are divided into smaller chunks for individual processing, and the results are then combined. It is useful for handling lengthy documents or datasets.
- **Reverse Chaining:** Reverse chaining starts with a desired output and works backward to determine the necessary inputs or steps to achieve it. It is great for problem-solving and debugging.

Each type of chaining caters to unique use cases, making it essential to choose the right one based on the task's complexity and requirements.

## Prerequisites

You need an IBM Cloud account to create a watsonx.ai project.

## Steps

### Step 1. Set up your environment

While you can choose from several tools, this tutorial walks you through how to set up an IBM account to use a Jupyter Notebook. Jupyter Notebooks are widely used within data science to combine code, text, images, and data visualizations to develop a well-formed analysis.

1. Log in to watsonx.ai using your IBM Cloud account.
2. Create a watsonx.ai project. Take note of the project ID in project > Manage > General > Project ID. You'll need this ID for this tutorial.
3. Create a Jupyter Notebook.

This step will open a Notebook environment where you can copy the code from this tutorial to implement a RAG application for Think 2024. Alternatively, you can download this notebook to your local system and upload it to your watsonx.ai project as an asset. To view more Granite tutorials, check out the IBM Granite Community. This tutorial is also available on Github.

### Step 2. Set up a Watson Machine Learning (WML) Service Instance and API Key

1. Create a Watson Machine Learning (WML) Service instance (choose the Lite plan, which is a free instance).
2. Generate an API Key in WML. Save this API key for use in this tutorial.
3. Associate the WML service to the project you created in watsonx.ai.

### Step 3. Understand the Use Case - Multi-Step Text Processing

We’ll process customer feedback by:
- Extracting keywords (Sequential Chaining).
- Generating a sentiment summary (Branching Chaining).
- Refining the sentiment summary if it doesn’t meet quality criteria (Iterative Chaining).

### Step 4. Install

We'll need a few libraries for this tutorial. Make sure to import the ones below, and if they're not installed, you can resolve this with a quick pip install.

In [24]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [25]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [26]:
!pip install langchain-ibm

### Step 5. Import Required Libraries

Start by importing the necessary libraries.

In [27]:
import os
from langchain_ibm import WatsonxLLM
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
import getpass

### Step 6. Set Up Credentials

Input your IBM Watson Machine Learning (WML) credentials and set the project_id.

In [28]:
'''
# Set up credentials
credentials = {
    "url": "https://us-south.ml.cloud.ibm.com",  # Replace with the correct region if needed
    "apikey": getpass.getpass("Please enter your WML API key (hit enter): ")
}

# Set up project_id
try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")
'''

'\n# Set up credentials\ncredentials = {\n    "url": "https://us-south.ml.cloud.ibm.com",  # Replace with the correct region if needed\n    "apikey": getpass.getpass("Please enter your WML API key (hit enter): ")\n}\n\n# Set up project_id\ntry:\n    project_id = os.environ["PROJECT_ID"]\nexcept KeyError:\n    project_id = input("Please enter your project_id (hit enter): ")\n'

### Step 7. Initialize the IBM LLM
Use the credentials to initialize the IBM Watson LLM.

In [30]:
# Step 2: Initialize the IBM LLM
llm = WatsonxLLM(
    model_id="ibm/granite-3-8b-instruct",
    url=credentials["url"],
    apikey=credentials["apikey"],
    project_id=project_id,
    params={
        "max_new_tokens": 150
    }
)

### Step 8. Define Prompt Templates
Define templates for each step in your chaining workflow.

In [31]:
# Step 3: Define Prompt Templates
# Prompt for extracting keywords
keyword_prompt = PromptTemplate(
    input_variables=["text"],
    template="Extract the most important keywords from the following text:\n{text}\n\nKeywords:"
)

# Prompt for generating sentiment summary
sentiment_prompt = PromptTemplate(
    input_variables=["keywords"],
    template="Using the following keywords, summarize the sentiment of the feedback:\nKeywords: {keywords}\n\nSentiment Summary:"
)

# Prompt for refining the summary
refine_prompt = PromptTemplate(
    input_variables=["sentiment_summary"],
    template="Refine the following sentiment summary to make it more concise and precise:\n{sentiment_summary}\n\nRefined Summary:"
)

### Step 9. Create Chains
Link the prompts to the initialized LLM using `LLMChain`.

In [32]:
# Step 4: Define Chains with Unique Keys
# Chain to extract keywords
keyword_chain = LLMChain(
    llm=llm,
    prompt=keyword_prompt,
    output_key="keywords"  # Unique key for extracted keywords
)

# Chain to generate sentiment summary
sentiment_chain = LLMChain(
    llm=llm,
    prompt=sentiment_prompt,
    output_key="sentiment_summary"  # Unique key for sentiment summary
)

# Chain to refine the sentiment summary
refine_chain = LLMChain(
    llm=llm,
    prompt=refine_prompt,
    output_key="refined_summary"  # Final refined output
)

### Step 10. Combine Chains
Use `SequentialChain` to combine the steps into a single workflow.

In [33]:
# Step 5: Combine Chains into a Sequential Workflow
workflow = SequentialChain(
    chains=[keyword_chain, sentiment_chain, refine_chain],
    input_variables=["text"],  # Initial input for the workflow
    output_variables=["refined_summary"]  # Final output of the workflow
)

### Step 11. Run the Workflow
Provide input text and execute the chain.

In [34]:
# Step 6: Example Input Text
feedback_text = """
I really enjoy the features of this app, but it crashes frequently, making it hard to use. 
The customer support is helpful, but response times are slow.
"""

# Step 7: Run the Workflow
result = workflow.run({"text": feedback_text})

# Step 8: Display the Output
print("Refined Sentiment Summary:")
print(result)  # Directly print the result since it is a string

/var/folders/4w/smh16qdx6l98q0534hr9v52r0000gn/T/ipykernel_3669/4193356551.py:8: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = workflow.run({"text": feedback_text})


Refined Sentiment Summary:


Users express dissatisfaction with the app due to frequent crashes and usability issues, exacerbated by slow customer support response times. The app's features, if present, may not be well-designed or user-friendly.


***Here will ge the summary of the output***

## Why Choose the Right Chaining Type?

- Task Complexity: Determine the number of steps required.
- Dependency: Assess whether outputs of steps depend on each other.
- Adaptability: Decide if the workflow needs dynamic or real-time changes.
- Data Modality: Ensure compatibility with text, images, or other data types.

Prompt chaining is a versatile technique for building sophisticated NLP workflows. In this tutorial, we explored various chaining types and demonstrated a generic example integrating multiple chaining approaches. By experimenting with these methods, you can unlock the full potential of language models for real-world applications.